# cancer-recon-apoptosis — Step 1: Boltz-2 Oracle Smoke Test (Colab)

**Goal:** verify Boltz-2 separates DR5 + TRAIL-binding loop (positive control) from DR5 + scrambled peptide (negative) by ≥2 kcal/mol.

**Decision rule:** if PASS → proceed to Step 2 (CellChat target shortlist). If FAIL → pivot per ASSESSMENT.md.

**Expected time:** 30–60 min total — ~10–15 min for weight download, ~5–10 min per prediction × 2.

**Required runtime:** Colab → Runtime → Change runtime type → **T4 GPU** (or better). Free tier T4 works.

## 1. Verify GPU

If `nvidia-smi` shows no GPU, go to *Runtime → Change runtime type → T4 GPU* and rerun.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"torch={torch.__version__} | cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device={torch.cuda.get_device_name(0)}")
    print(f"  VRAM={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configure repo URL and clone

Set `REPO_URL` to your GitHub repo once you've pushed it. Default below uses the canonical `AnshumanAtrey/cancer-recon-apoptosis` — change if your username is different.

In [ ]:
import os
from pathlib import Path

# ← EDIT this if your GitHub repo is at a different URL
REPO_URL = "https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git"
REPO_DIR = Path("/content/cancer-recon-apoptosis")

if REPO_DIR.exists():
    print(f"  repo already cloned at {REPO_DIR} — pulling latest...")
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n  cwd = {os.getcwd()}")
!ls -la

## 3. Install Boltz-2

First-time install pulls ~10 GB of model weights on first inference (downloaded lazily). The `pip install` itself is fast (~2 min).

In [ ]:
!pip install --quiet boltz biopython

In [ ]:
!which boltz && boltz --help | head -20

## 4. Run the Step-1 smoke test

This runs `scripts/01_boltz_smoketest.py` which:
1. Loads the four reference sequences from `data/sequences/`.
2. Writes Boltz YAML inputs for the two complexes (`DR5+TRAIL_loop`, `DR5+scrambled`).
3. Invokes `boltz predict --use_msa_server` for each. **First run downloads ~10 GB of weights.**
4. Parses the affinity output and reports the ΔG gap.

Expected total time: ~15–40 min.

In [ ]:
!python scripts/01_boltz_smoketest.py

## 5. Inspect the raw outputs

Boltz writes per-complex outputs into `runs/step1_boltz/{positive,negative}/`. We look at the affinity JSON and the predicted structure file.

In [ ]:
import json
from pathlib import Path

RUN_DIR = Path("runs/step1_boltz")

print("=== Output tree ===")
!find runs/step1_boltz -maxdepth 4 -type f | head -40

print("\n=== Affinity JSONs ===")
for j in RUN_DIR.rglob("affinity*.json"):
    print(f"\n--- {j.relative_to(RUN_DIR)} ---")
    print(json.dumps(json.loads(j.read_text()), indent=2))

In [ ]:
# Compute the ΔG gap from the raw affinity values, redundant with the script's
# decision but useful for sanity-checking.
import json
from pathlib import Path

def first_affinity(d):
    for j in Path(d).rglob("affinity*.json"):
        return json.loads(j.read_text())
    return None

pos = first_affinity("runs/step1_boltz/positive")
neg = first_affinity("runs/step1_boltz/negative")

if pos and neg:
    # Boltz affinity_pred_value is a pKd-like log value; ΔG ≈ -1.364 * pKd at 298 K
    pos_dg = -1.364 * float(pos['affinity_pred_value'])
    neg_dg = -1.364 * float(neg['affinity_pred_value'])
    gap = neg_dg - pos_dg  # if positive binds tighter, ΔG_pos is more negative → gap is positive
    print(f"  ΔG(positive DR5+TRAIL_loop)  = {pos_dg:+.2f} kcal/mol")
    print(f"  ΔG(negative DR5+scrambled)   = {neg_dg:+.2f} kcal/mol")
    print(f"  gap = ΔG(neg) − ΔG(pos)      = {gap:+.2f} kcal/mol")
    if gap >= 2.0:
        print("\n  ✅ PASS — oracle separates signal from noise. Proceed to Step 2.")
    else:
        print("\n  ❌ FAIL — see ASSESSMENT.md Day-1 kill criteria.")
else:
    print("  ⚠️ affinity JSON not found in run outputs — inspect runs/step1_boltz/ manually")

## 6. Save outputs to Google Drive (optional)

Colab sessions are ephemeral — when the runtime resets, `/content/` is wiped. To persist the smoke test results, mount Drive and copy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = Path('/content/drive/MyDrive/cancer-recon-apoptosis/step1_boltz')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

!cp -r runs/step1_boltz/* '{DRIVE_OUT}/'
print(f"\n  saved → {DRIVE_OUT}")
!ls -la '{DRIVE_OUT}'

## 7. Optional — push results back to GitHub

If you want results committed back to the repo, run the cell below. **Skip if you don't want runs/ in version control** (the project's `.gitignore` already excludes `runs/`).

In [ ]:
# ⚠️ Only run if you want to commit smoke-test results to the repo.
# `.gitignore` excludes runs/ by default — uncomment if you want to override.

# !git config user.name 'Anshuman Atrey'
# !git config user.email 'YOUR_EMAIL@example.com'
# !git add -f runs/step1_boltz/  # -f overrides .gitignore
# !git commit -m 'step 1: Boltz-2 smoke test results'
# !git push  # will prompt for GitHub PAT
print("  (skipped — uncomment cell to push)")

---

## If the smoke test PASSED

1. Update `runs/step1_local/SMOKETEST_RESULTS.md` with the real cloud numbers.
2. Proceed to **Step 2** — CellChat cancer-restricted target shortlist (`scripts/02_cellchat_targets.R`, to be written).

## If the smoke test FAILED (gap < 2 kcal/mol)

See `ASSESSMENT.md` Day-1 kill criteria. Pivot options:
1. **AlphaFold 3 Server** — submit complexes via web API for cross-check.
2. **ABFE pipeline** — Recursion Oct 2025 pipeline atop Boltz-2 ensemble.
3. **OpenMM MD refinement** — 5 ns MD after Boltz-2 prediction for refined ΔG.

## Common Colab issues

- **`boltz: command not found`** → restart runtime after `pip install` and re-run cells from §3.
- **OOM error** → switch to High-RAM runtime, or use `--diffusion_samples 1` instead of 5.
- **MSA server timeout** → re-run; ColabFold MSA queue is sometimes congested. Or pre-compute MSAs locally and pass via `--msa_dir`.
- **Runtime disconnected** → free tier idle limit. Keep tab focused, or upgrade to Colab Pro ($10/mo) for A100.